In [ ]:
from google.colab import drive

# 다시 마운트
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
cd drive/My\ Drive/"Path to your folder" # Path to your folder 부분 입력

/content/drive/My Drive/RND/NIA


In [ ]:
import pandas as pd
import json
import os

# Step 2: 스프레드시트 읽기
df = pd.read_excel('summary_data_koalpaca.xlsx')   # 폴더 내 summary_data_koalpaca.xlsx 파일 배치
df = df.iloc[:1278]  # training 에 필요한 데이터 분할

# Step 3: JSON 변환
json_data = {"data": []}

for index, row in df.iterrows():
    json_data["data"].append({
        "instruction": row["instruction"],
        "input": row["input"],
        "output": row["output"]
    })

# Step 4: JSON 파일 저장
json_file_path = os.path.join('koalpaca_data.json')
with open(json_file_path, 'w', encoding='utf-8') as json_file:
    json.dump(json_data, json_file, ensure_ascii=False, indent=4)

print(f'JSON 파일이 저장되었습니다: {json_file_path}')

In [ ]:
import torch

In [ ]:
from datetime import datetime

# 타임스탬프 출력
print("실행 시각:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

# CPU 사양
print("\n1. CPU 사양")
!lscpu

# GPU 사양 (GPU가 연결된 경우에만 출력됩니다)
print("\n2. GPU 사양")
!nvidia-smi

# RAM 사양
print("\n3. RAM 사양")
!free -h

# HDD 용량
print("\n4. HDD 용량")
!df -h

# OS 버전
print("\n5. OS 버전")
!lsb_release -a

# pytorch 버전
print("\n6. 프레임워크 버전")
print(torch.__version__)

실행 시각: 2024-12-11 04:30:48

1. CPU 사양
Architecture:             x86_64
  CPU op-mode(s):         32-bit, 64-bit
  Address sizes:          46 bits physical, 48 bits virtual
  Byte Order:             Little Endian
CPU(s):                   12
  On-line CPU(s) list:    0-11
Vendor ID:                GenuineIntel
  Model name:             Intel(R) Xeon(R) CPU @ 2.20GHz
    CPU family:           6
    Model:                85
    Thread(s) per core:   2
    Core(s) per socket:   6
    Socket(s):            1
    Stepping:             7
    BogoMIPS:             4400.47
    Flags:                fpu vme de pse tsc msr pae mce cx8 apic sep mtrr pge mca cmov pat pse36 cl
                          flush mmx fxsr sse sse2 ss ht syscall nx pdpe1gb rdtscp lm constant_tsc re
                          p_good nopl xtopology nonstop_tsc cpuid tsc_known_freq pni pclmulqdq ssse3
                           fma cx16 pcid sse4_1 sse4_2 x2apic movbe popcnt aes xsave avx f16c rdrand
                         

In [ ]:
!pip install -q -U bitsandbytes
!pip install -q -U git+https://github.com/huggingface/transformers.git
!pip install -q -U git+https://github.com/huggingface/peft.git
!pip install -q -U git+https://github.com/huggingface/accelerate.git
!pip install -q datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 MB 33.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 29.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 18.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver d

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "EleutherAI/polyglot-ko-12.8b"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map={"":0})

# 모델을 GPU로 이동 (필요 시)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("4bit 양자화된 모델과 토크나이저를 성공적으로 불러왔습니다.")

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/174 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.65M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/204 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/678 [00:00<?, ?B/s]

...

model-00028-of-00028.safetensors:   0%|          | 0.00/518M [00:00<?, ?B/s]

The `GPTNeoXSdpaAttention` class is deprecated in favor of simply modifying the `config._attn_implementation`attribute of the `GPTNeoXAttention` class! It will be removed in v4.48


Loading checkpoint shards:   0%|          | 0/28 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

4bit 양자화된 모델과 토크나이저가 성공적으로 불러와졌습니다.


In [ ]:
from peft import prepare_model_for_kbit_training

model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

In [ ]:
def print_trainable_parameters(model):
    """
    Prints the number of trainable parameters in the model.
    """
    trainable_params = 0
    all_param = 0
    for _, param in model.named_parameters():
        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
    print(
        f"trainable params: {trainable_params} || all params: {all_param} || trainable%: {100 * trainable_params / all_param}"
    )

In [ ]:
from peft import LoraConfig, get_peft_model

config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["query_key_value"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, config)
print_trainable_parameters(model)

trainable params: 6553600 || all params: 6608701440 || trainable%: 0.09916622894073424


In [ ]:
from datasets import load_dataset
from datasets import Dataset

import json

# JSON 파일 경로
json_file_path = "koalpaca_data.json"

# JSON 파일 읽기
with open(json_file_path, 'r') as f:
    data_dict = json.load(f)

# 데이터셋 생성
dataset = load_dataset("json", data_files=json_file_path, field="data")

# 데이터셋 확인 (선택사항)
print(dataset)

# 데이터셋 텍스트 형식으로 변환
data = dataset.map(
     lambda x:
     {'text': f"### 명령어: {x['instruction']}\n\n###맥락: {x['input']}\n\n### 답변: {x['output']}<|endoftext|>" }
     if x['input'] else
     {'text':f"### 명령어: {x['instruction']}\n\n### 답변: {x['output']}<|endoftext|>"},
)

data = data.map(lambda samples: tokenizer(samples["text"]), batched=True)

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 1278
    })
})


Map:   0%|          | 0/1278 [00:00<?, ? examples/s]

Map:   0%|          | 0/1278 [00:00<?, ? examples/s]

In [ ]:
!nvidia-smi

Wed Dec 11 04:35:21 2024       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.104.05             Driver Version: 535.104.05   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off | 00000000:00:04.0 Off |                    0 |
| N/A   33C    P0              50W / 400W |   8755MiB / 40960MiB |      0%      Default |
|                                         |                      |             Disabled |
+-----------------------------------------+----------------------+--

In [ ]:
import torch

# GPU 메모리 해제
torch.cuda.empty_cache()

# 추가적으로 메모리 사용 확인
print(torch.cuda.memory_summary(device=None, abbreviated=False))

|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |   7560 MiB |   7829 MiB |  32903 MiB |  25343 MiB |
|       from large pool |   7506 MiB |   7800 MiB |  32844 MiB |  25337 MiB |
|       from small pool |     53 MiB |     53 MiB |     59 MiB |      5 MiB |
|---------------------------------------------------------------------------|
| Active memory         |   7560 MiB |   7829 MiB |  32903 MiB |  25343 MiB |
|       from large pool |   7506 MiB |   7800 MiB |  32844 MiB |

In [ ]:
import transformers

# needed for gpt-neo-x tokenizer
tokenizer.pad_token = tokenizer.eos_token

trainer = transformers.Trainer(
    model=model,
    train_dataset=data['train'],
    args=transformers.TrainingArguments(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1,
        # warmup_steps=200,
        max_steps=500,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=10,
        output_dir="outputs",
        optim="paged_adamw_8bit"
    ),
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False),
)
model.config.use_cache = False  # silence the warnings. Please re-enable for inference!
trainer.train()

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter, or press ctrl+c to quit:

 ··········


wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


/usr/local/lib/python3.10/dist-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,5.626700
20,4.293300
30,3.576800
40,3.240900
50,2.970100
60,2.851300
70,2.629600
80,2.717200
90,2.554100
100,2.524000


TrainOutput(global_step=500, training_loss=2.5288353309631346, metrics={'train_runtime': 4342.0309, 'train_samples_per_second': 0.115, 'train_steps_per_second': 0.115, 'total_flos': 4.9105052047202304e+17, 'train_loss': 2.5288353309631346, 'epoch': 0.39123630672926446})

In [ ]:
# 모델을 저장할 디렉토리 지정
output_dir = "./koalpaca_save"

# 모델 저장
model.save_pretrained(output_dir)

# 토크나이저 저장 (필요 시)
tokenizer.save_pretrained(output_dir)

# 기존 모델에서 설정 불러오기
config = model.config

# config.json 파일 수동 저장
config.save_pretrained(output_dir)

print(f"모델이 {output_dir}에 저장되었습니다.")

모델이 ./koalpaca_save에 저장되었습니다.
